In [1]:
# 导入所需的库
import pandas as pd
import numpy as np
import math
import openpyxl
import warnings
from collections import Counter
from pathlib import Path

warnings.filterwarnings('ignore')

In [2]:
# 输入相关配置参数

# 配置根目录
ROOT_DIR = Path(r"E:\pycharm all files\眼动数据处理\Data")

# 阶段文件映射
PHASE_FILES = {
    "1.qf.xlsx": "起飞阶段",
    "2.zw1.xlsx": "第1次转弯",
    "3.zw2.xlsx": "第2次转弯",
    "4.xh.xlsx": "巡航阶段",
    "5.zw3.xlsx": "第3次转弯",
    "6.zw4.xlsx": "第4次转弯",
    "7.jl.xlsx": "降落阶段"
}

# 无效数据记录（根据您提供的信息）
INVALID_DATA = {
    # A组无效数据
    ('A', '女', '杨珊珊'): [3],
    ('A', '男', '徐梓军'): [1, 2, 6],
    ('A', '男', '曾奇'): [1],
    ('A', '男', '朱祖恩'): [2, 4],
    ('A', '女', '王蓉蓉'): [7],
    ('A', '男', '王柯辰'): [1, 2, 3, 4, 5, 6, 7],
    # B组无效数据
    ('B', '男', '冯科嘉'): [1],
    ('B', '男', '毛华浩'): [6],
    ('B', '男', '王子铭'): [6],
    ('B', '男', '祝鹏程'): [3],
    ('B', '男', '程腾宇'): [1, 2]
}

In [3]:
# 在非眨眼处插入AOI5

def preprocess_aoi_data(data):
    """
    预处理AOI数据：当BKID=0时，将AOI列的空值改为5

    参数:
    data -- 包含眼动数据的DataFrame

    返回:
    处理后的DataFrame和状态信息
    """
    # 检查是否有AOI列
    if 'AOI' not in data.columns:
        return data, "跳过处理（无AOI列）"

    # 检查是否有BKID列
    if 'BKID' not in data.columns:
        return data, "跳过处理（无BKID列）"

    # 确保BKID是数值类型
    if not pd.api.types.is_numeric_dtype(data['BKID']):
        data['BKID'] = pd.to_numeric(data['BKID'], errors='coerce')

    # 条件修改：当BKID=0且AOI为空时，将AOI设为5
    condition = (data['BKID'] == 0) & (data['AOI'].isna())
    data.loc[condition, 'AOI'] = 'AOI 5'

    return data, "处理成功"


In [4]:
# def filter_outliers(data):
#     """
#     异常值筛选函数
# 
#     参数:
#     data -- DataFrame，包含至少 AOI 和 FPOGS 列
# 
#     返回:
#     DataFrame -- 筛选后的有效数据
#     """
#     df = data.copy()
# 
#     # 1. AOI合法性（只允许1~5）
#     df = df[df['AOI'].between(1, 5)]
# 
#     # 2. 时间戳非负
#     df = df[df['FPOGS'] >= 0]
# 
#     # 3. 去除重复时间戳
#     df = df.drop_duplicates(subset=['FPOGS'])
# 
#     # 4. 注视时长过滤（可选，只有有Duration列才生效）
#     if 'Duration' in df.columns:
#         df = df[(df['Duration'] >= 60) & (df['Duration'] <= 2000)]
# 
#     return df


In [5]:
def calculate_attention_metrics(data, phase_name):
    """
    计算特定阶段的注意力指标

    参数:
    data -- 包含眼动数据的DataFrame
    phase_name -- 阶段名称（用于结果标识）

    返回:
    包含计算指标的字典
    """
    try:
        # 1. 数据预处理
        # 确保FPOGS列是数值类型
        if not pd.api.types.is_numeric_dtype(data['FPOGS']):
            data['FPOGS'] = pd.to_numeric(data['FPOGS'], errors='coerce')

        # 确保数据按时间顺序排列
        data = data.sort_values(by='FPOGS')

        # 处理AOI列 - 确保是整数类型
        if not pd.api.types.is_integer_dtype(data['AOI']):
            # 尝试提取数字部分（处理"AOI 1"格式）
            try:
                data['AOI'] = data['AOI'].astype(str).str.extract(r'(\d+)').astype(float).fillna(-1).astype(int)
            except:
                data['AOI'] = pd.to_numeric(data['AOI'], errors='coerce').fillna(-1).astype(int)

        # 过滤有效AOI数据（假设无效AOI标记为-1）
        valid_data = data[data['AOI'] != -1].copy()

        # # 调用异常值筛选
        # valid_data = filter_outliers(data)

        # 添加诊断信息
        total_points = len(data)  # 总注视点数
        valid_points = len(valid_data)  # 有效注视点数
        valid_ratio = valid_points / total_points if total_points > 0 else 0  # 有效率
        unique_aois = valid_data['AOI'].nunique()  # 唯一AOI数量

        # 初始化结果
        result = {
            '阶段': phase_name,
            'AOI转换次数': np.nan,
            '静态注视熵(SGE)': np.nan,
            '眼跳注视熵(GTE)': np.nan,
        }

        # 2. 计算AOI转换次数（只需要至少1个有效点）
        if valid_points >= 1:
            aoi_sequence = valid_data['AOI'].values
            transitions = 0
            for i in range(1, len(aoi_sequence)):
                if aoi_sequence[i] != aoi_sequence[i - 1]:
                    transitions += 1
            result['AOI转换次数'] = transitions

        # 3. 计算静态注视熵 (SGE)（只需要至少1个有效点）
        if valid_points >= 1:
            # 计算每个AOI的注视点数量
            aoi_counts = valid_data['AOI'].value_counts()
            # 计算每个AOI的概率
            total_fixations = len(valid_data)
            probabilities = aoi_counts / total_fixations

            # 计算熵值
            sge = 0
            for p in probabilities:
                if p > 0:  # 避免log(0)错误
                    sge -= p * math.log2(p)
            result['静态注视熵(SGE)'] = sge

        # 4. 计算眼跳注视熵 (GTE)（需要至少2个有效点）
        if valid_points >= 2:
            # 创建转移矩阵
            transition_matrix = {}
            unique_aois = sorted(valid_data['AOI'].unique())
            for aoi in unique_aois:
                transition_matrix[aoi] = Counter()

            # 填充转移矩阵
            aoi_sequence = valid_data['AOI'].values
            for i in range(1, len(aoi_sequence)):
                from_aoi = aoi_sequence[i - 1]
                to_aoi = aoi_sequence[i]
                transition_matrix[from_aoi][to_aoi] += 1

            # 计算条件熵
            gte = 0
            for from_aoi, transitions_count in transition_matrix.items():
                total_transitions = sum(transitions_count.values())
                if total_transitions == 0:
                    continue

                # 计算从当前AOI转移的条件熵
                conditional_entropy = 0
                for count in transitions_count.values():
                    p = count / total_transitions
                    if p > 0:
                        conditional_entropy -= p * math.log2(p)

                # 加权平均
                p_from = aoi_counts.get(from_aoi, 0) / total_fixations
                gte += p_from * conditional_entropy
            result['眼跳注视熵(GTE)'] = gte

        # 返回结果
        return result

    except Exception as e:
        print(f"在计算{phase_name}阶段指标时出错: {str(e)}")
        import traceback
        traceback.print_exc()
        return {
            '阶段': phase_name,
            'AOI转换次数': np.nan,
            '静态注视熵(SGE)': np.nan,
            '眼跳注视熵(GTE)': np.nan,
            '错误信息': str(e)
        }

In [6]:
def process_all_data():
    """
    处理所有参与者的所有数据
    """
    results = []
    total_files = 0
    processed_files = 0
    skipped_files = 0
    total_eye_duration = 0  # 累计总时长（毫秒）

    print(f"开始处理根目录: {ROOT_DIR}")

    # 遍历组别 (A, B)
    for group_dir in ROOT_DIR.iterdir():
        if not group_dir.is_dir() or group_dir.name not in ['A', 'B']:
            continue

        print(f"\n处理组别: {group_dir.name}")

        # 遍历性别 (男, 女)
        for gender_dir in group_dir.iterdir():
            if not gender_dir.is_dir() or gender_dir.name not in ['男', '女']:
                continue

            print(f"\n处理性别: {gender_dir.name}")

            # 遍历参与者
            for participant_dir in gender_dir.iterdir():
                if not participant_dir.is_dir():
                    continue

                print(f"\n处理参与者: {participant_dir.name}")

                # 遍历天数 (1-7)
                for day in range(1, 8):
                    day_dir = participant_dir / str(day)

                    # 检查是否为无效数据
                    if (group_dir.name, gender_dir.name, participant_dir.name) in INVALID_DATA:
                        if day in INVALID_DATA[(group_dir.name, gender_dir.name, participant_dir.name)]:
                            print(f"跳过无效数据: {participant_dir.name} 第{day}天")
                            skipped_files += 7  # 跳过该天的所有7个文件
                            continue

                    if not day_dir.exists() or not day_dir.is_dir():
                        print(f"天数目录不存在: {day_dir}")
                        skipped_files += 7
                        continue

                    print(f"处理第{day}天数据")

                    # 处理每个阶段文件
                    for phase_file, phase_name in PHASE_FILES.items():
                        file_path = day_dir / phase_file
                        total_files += 1

                        if not file_path.exists():
                            print(f"文件不存在: {file_path}")
                            skipped_files += 1
                            continue

                        try:
                            # 读取Excel文件
                            data = pd.read_excel(file_path)

                            # 预处理AOI数据
                            preprocessed_data, preprocess_status = preprocess_aoi_data(data)

                            if "跳过" in preprocess_status:
                                print(f"{preprocess_status}: {file_path}")
                                skipped_files += 1
                                continue

                            # 计算指标
                            metrics = calculate_attention_metrics(preprocessed_data, phase_name)

                            # 计算当前文件时长
                            if 'FPOGS' in preprocessed_data.columns:
                                fpogs = pd.to_numeric(preprocessed_data['FPOGS'], errors='coerce').dropna()
                                if len(fpogs) > 1:
                                    file_duration = fpogs.max() - fpogs.min()
                                elif len(fpogs) == 1:
                                    file_duration = 0
                                else:
                                    file_duration = 0
                                total_eye_duration += file_duration

                            # 添加元数据
                            metrics.update({
                                '组别': group_dir.name,
                                '性别': gender_dir.name,
                                '被试者': participant_dir.name,
                                '天数': day,
                                '文件路径': str(file_path)
                            })

                            results.append(metrics)
                            processed_files += 1

                        except Exception as e:
                            print(f"处理文件 {file_path} 时出错: {str(e)}")
                            results.append({
                                '组别': group_dir.name,
                                '性别': gender_dir.name,
                                '被试者': participant_dir.name,
                                '天数': day,
                                '阶段': phase_name,
                                '错误信息': f"文件处理错误: {str(e)}"
                            })
                            skipped_files += 1

    print(f"\n处理完成! 总文件数: {total_files}, 成功处理: {processed_files}, 跳过/失败: {skipped_files}")
    print(f"所有被试所有阶段的总眼动数据时长: {total_eye_duration} ms")
    return pd.DataFrame(results)

In [7]:
def create_excel_report(results_df, output_file):
    """
    创建格式化的Excel报告

    参数:
    results_df -- 包含结果的数据框
    output_file -- 输出文件路径
    """
    try:
        # 创建Excel写入器
        with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
            # 写入主要结果
            results_df.to_excel(writer, sheet_name='结果摘要', index=False)

            # 获取工作簿和工作表对象
            workbook = writer.book
            sheet = writer.sheets['结果摘要']

            # 设置列宽
            for column in sheet.columns:
                max_length = 0
                column_letter = openpyxl.utils.get_column_letter(column[0].column)

                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass

                adjusted_width = (max_length + 2)
                sheet.column_dimensions[column_letter].width = adjusted_width

            print(f"报告已保存至: {output_file}")
            return True
    except Exception as e:
        print(f"创建Excel报告时出错: {str(e)}")
        # 如果格式化失败，尝试简单保存
        try:
            results_df.to_excel(output_file, index=False)
            print(f"简单报告已保存至: {output_file}")
            return True
        except:
            print(f"无法保存报告: {str(e)}")
            return False

In [8]:
# 主程序
if __name__ == "__main__":
    print("开始处理所有眼动数据...")

    # 配置输出文件
    OUTPUT_EXCEL = "眼动数据预处理文件.xlsx"

    # 处理所有数据
    results_df = process_all_data()

    if results_df.empty:
        print("未找到有效数据处理，请检查配置路径")
    else:
        print(f"\n数据处理完成，共处理 {len(results_df)} 条记录")

        # 保存详细结果
        if create_excel_report(results_df, OUTPUT_EXCEL):
            print(f"详细结果已保存至: {OUTPUT_EXCEL}")
        else:
            print("详细结果保存失败")

    print("分析完成！")

开始处理所有眼动数据...
开始处理根目录: E:\pycharm all files\眼动数据处理\Data

处理组别: A

处理性别: 女

处理参与者: 付瑞晗
处理第1天数据
处理第2天数据
处理第3天数据
处理第4天数据
处理第5天数据
处理第6天数据
处理第7天数据

处理参与者: 徐子焰
处理第1天数据
处理第2天数据
处理第3天数据
处理第4天数据
处理第5天数据
处理第6天数据
处理第7天数据

处理参与者: 徐文婷
处理第1天数据
处理第2天数据
处理第3天数据
处理第4天数据
处理第5天数据
处理第6天数据
处理第7天数据

处理参与者: 杨珊珊
处理第1天数据
处理第2天数据
跳过无效数据: 杨珊珊 第3天
处理第4天数据
处理第5天数据
处理第6天数据
处理第7天数据

处理参与者: 王蓉蓉
处理第1天数据
处理第2天数据
处理第3天数据
处理第4天数据
处理第5天数据
处理第6天数据
跳过无效数据: 王蓉蓉 第7天

处理参与者: 达格吉
处理第1天数据
处理第2天数据
处理第3天数据
处理第4天数据
处理第5天数据
处理第6天数据
处理第7天数据

处理性别: 男

处理参与者: 刘子皓
处理第1天数据
处理第2天数据
处理第3天数据
处理第4天数据
处理第5天数据
处理第6天数据
处理第7天数据

处理参与者: 张渊博
处理第1天数据
处理第2天数据
处理第3天数据
处理第4天数据
处理第5天数据
处理第6天数据
处理第7天数据

处理参与者: 徐梓军
跳过无效数据: 徐梓军 第1天
跳过无效数据: 徐梓军 第2天
处理第3天数据
处理第4天数据
处理第5天数据
跳过无效数据: 徐梓军 第6天
处理第7天数据

处理参与者: 曾奇
跳过无效数据: 曾奇 第1天
处理第2天数据
处理第3天数据
处理第4天数据
处理第5天数据
处理第6天数据
处理第7天数据

处理参与者: 朱祖恩
处理第1天数据
跳过无效数据: 朱祖恩 第2天
处理第3天数据
跳过无效数据: 朱祖恩 第4天
处理第5天数据
处理第6天数据
处理第7天数据

处理参与者: 李嘉文
处理第1天数据
处理第2天数据
处理第3天数据
处理第4天数据
处理第5天数据
处理第6天数据
处理第7天数据

处理参与者: 李国荣
处理第1天数据
处理第2天数据
处理第3天数据
处理第